In [1]:
# 从纯文本导入 英文 分节经文 到数据库

import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

def parse_filename(filename):
    name = os.path.splitext(filename)[0].replace("_en", "")
    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)
    chapter = int(match.group(2))
    book_id = get_book_id(book_abbr)
    return book_abbr, chapter, book_id


def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_en FROM verse WHERE id = ?", (verse_id,)
        )
        row = cur.fetchone()

        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            print(f"➕ 新增 {verse_id}")

        elif row[0] == text_en:
            pass

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_en = ? WHERE id = ?",
                    (text_en, verse_id)
                )
                print(f"♻️  已覆盖 {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 导入完成")


# 只问文件名
if __name__ == "__main__":
    filename = input("请输入文件名（如 Mt.1.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入文件名（如 Mt.1.txt）：2K_4_en.txt
➕ 新增 2K.4.1
➕ 新增 2K.4.2
➕ 新增 2K.4.3
➕ 新增 2K.4.4
➕ 新增 2K.4.5
➕ 新增 2K.4.6
➕ 新增 2K.4.7
➕ 新增 2K.4.8
➕ 新增 2K.4.9
➕ 新增 2K.4.10
➕ 新增 2K.4.11
➕ 新增 2K.4.12
➕ 新增 2K.4.13
➕ 新增 2K.4.14
➕ 新增 2K.4.15
➕ 新增 2K.4.16
➕ 新增 2K.4.17
➕ 新增 2K.4.18
➕ 新增 2K.4.19
➕ 新增 2K.4.20
➕ 新增 2K.4.21
➕ 新增 2K.4.22
➕ 新增 2K.4.23
➕ 新增 2K.4.24
➕ 新增 2K.4.25
➕ 新增 2K.4.26
➕ 新增 2K.4.27
➕ 新增 2K.4.28
➕ 新增 2K.4.29
➕ 新增 2K.4.30
➕ 新增 2K.4.31
➕ 新增 2K.4.32
➕ 新增 2K.4.33
➕ 新增 2K.4.34
➕ 新增 2K.4.35
➕ 新增 2K.4.36
➕ 新增 2K.4.37
➕ 新增 2K.4.38
➕ 新增 2K.4.39
➕ 新增 2K.4.40
➕ 新增 2K.4.41
➕ 新增 2K.4.42
➕ 新增 2K.4.43
➕ 新增 2K.4.44

✅ 2K.4 导入完成


In [7]:
# import_cn.py
# 从纯文本导入中文分节经文（text_cn）
# 文件名规范：2K_4_cn.txt

import sqlite3
import os
import re

# ==========================
# 数据库连接
# ==========================
conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ==========================
# 从 book 表获取 book_id
# ==========================
def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

# ==========================
# 解析文件名（2K_4_cn.txt）
# ==========================
def parse_filename(filename):
    name = os.path.splitext(filename)[0]          # 去掉 .txt
    name = name.replace("_cn", "")                 # 去掉 _cn

    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)   # 2K
    chapter = int(match.group(2))  # 4
    book_id = get_book_id(book_abbr)

    return book_abbr, chapter, book_id

# ==========================
# 导入中文经文
# ==========================
def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_cn in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_cn FROM verse WHERE id = ?",
            (verse_id,)
        )
        row = cur.fetchone()

        # verse 不存在（极少见）
        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, "", text_cn))
            print(f"➕ 新增 {verse_id}")

        elif row[0] is None or row[0] == "":
            cursor.execute(
                "UPDATE verse SET text_cn = ? WHERE id = ?",
                (text_cn, verse_id)
            )
            print(f"➕ 写入 text_cn: {verse_id}")

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖中文译文？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_cn = ? WHERE id = ?",
                    (text_cn, verse_id)
                )
                print(f"♻️  已覆盖 text_cn: {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 中文经文导入完成")

# ==========================
# 主程序
# ==========================
if __name__ == "__main__":
    filename = input("请输入中文经文文件名（如 2K_4_cn.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入中文经文文件名（如 2K_4_cn.txt）：2K_4_cn.txt
➕ 写入 text_cn: 2K.4.1
➕ 写入 text_cn: 2K.4.2
➕ 写入 text_cn: 2K.4.3
➕ 写入 text_cn: 2K.4.4
➕ 写入 text_cn: 2K.4.5
➕ 写入 text_cn: 2K.4.6
➕ 写入 text_cn: 2K.4.7
➕ 写入 text_cn: 2K.4.8
➕ 写入 text_cn: 2K.4.9
➕ 写入 text_cn: 2K.4.10
➕ 写入 text_cn: 2K.4.11
➕ 写入 text_cn: 2K.4.12
➕ 写入 text_cn: 2K.4.13
➕ 写入 text_cn: 2K.4.14
➕ 写入 text_cn: 2K.4.15
➕ 写入 text_cn: 2K.4.16
➕ 写入 text_cn: 2K.4.17
➕ 写入 text_cn: 2K.4.18
➕ 写入 text_cn: 2K.4.19
➕ 写入 text_cn: 2K.4.20
➕ 写入 text_cn: 2K.4.21
➕ 写入 text_cn: 2K.4.22
➕ 写入 text_cn: 2K.4.23
➕ 写入 text_cn: 2K.4.24
➕ 写入 text_cn: 2K.4.25
➕ 写入 text_cn: 2K.4.26
➕ 写入 text_cn: 2K.4.27
➕ 写入 text_cn: 2K.4.28
➕ 写入 text_cn: 2K.4.29
➕ 写入 text_cn: 2K.4.30
➕ 写入 text_cn: 2K.4.31
➕ 写入 text_cn: 2K.4.32
➕ 写入 text_cn: 2K.4.33
➕ 写入 text_cn: 2K.4.34
➕ 写入 text_cn: 2K.4.35
➕ 写入 text_cn: 2K.4.36
➕ 写入 text_cn: 2K.4.37
➕ 写入 text_cn: 2K.4.38
➕ 写入 text_cn: 2K.4.39
➕ 写入 text_cn: 2K.4.40
➕ 写入 text_cn: 2K.4.41
➕ 写入 text_cn: 2K.4.42
➕ 写入 text_cn: 2K.4.43
➕ 写入 text_cn: 2K.4.44

✅ 

In [3]:
# 清空数据

import sqlite3

DB_PATH = "db/bible.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DELETE FROM verse;")
conn.commit()
conn.close()

print("✅ verse 表数据已全部清空")

✅ verse 表数据已全部清空
